# Whisper LoRA pilot on Kaggle

This notebook trains only PEFT/LoRA adapters. The baseline, tokenizer, and external 96-utterance test remain protected.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

REPO_URL = os.environ.get('REPO_URL', 'https://github.com/mehdiyds/tunisian-speech-ai.git')
REPO_BRANCH = os.environ.get('REPO_BRANCH', 'codex/smoke-test-whisper-fixes')
REPO_DIR = Path('/kaggle/working/tunisian-speech-ai')

# The Kaggle Dataset contains data only; the code is cloned separately.
if not (REPO_DIR / 'scripts' / 'train_lora.py').exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

required_files = [
    REPO_DIR / 'requirements.txt',
    REPO_DIR / 'scripts' / 'train_lora.py',
    REPO_DIR / 'src' / 'data' / 'validate_dataset.py',
]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError('Incomplete repository clone; missing: ' + ', '.join(missing))

sys.path.insert(0, str(REPO_DIR))
print('Repository:', REPO_DIR)
print('Branch:', REPO_BRANCH)
print('Training script:', REPO_DIR / 'scripts' / 'train_lora.py')

In [ ]:
# Install versions compatible with the pinned Transformers/PEFT stack.
# Kaggle's preinstalled packages are intentionally not replaced wholesale.
%pip install -q --upgrade --force-reinstall --no-deps \
    transformers==4.48.3 \
    tokenizers==0.21.0 \
    huggingface-hub==0.27.1 \
    accelerate==1.2.1 \
    peft==0.14.0 \
    datasets==3.2.0 \
    evaluate==0.4.3 \
    jiwer==3.0.5 \
    rapidfuzz==3.11.0 \
    soundfile==0.13.0 \
    PyYAML==6.0.2
print('Pinned project dependencies installed.')
print('Restart the Kaggle kernel now, then run all cells from the beginning.')

In [ ]:
# Check the Kaggle PyTorch stack before importing Transformers.
# Do not install torch/torchvision from requirements.txt on Kaggle.
import torch

if not hasattr(torch, '_utils'):
    raise RuntimeError(
        'Kaggle PyTorch is corrupted in this session. Use Session > Factory reset, '
        'then run this notebook again. Do not run pip install -r requirements.txt.'
    )

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('CUDA unavailable. Enable a GPU accelerator in Kaggle.')
print('GPU:', torch.cuda.get_device_name(0))

# Import only after the PyTorch integrity check.
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from jiwer import wer, cer
print('PyTorch and training imports: PASSED')

In [ ]:
# Check the Kaggle PyTorch stack before importing Transformers.
# Do not install torch/torchvision from requirements.txt on Kaggle.
import shutil
import subprocess
import torch

if not hasattr(torch, '_utils'):
    raise RuntimeError(
        'Kaggle PyTorch is corrupted in this session. Use Session > Factory reset, '
        'then run this notebook again. Do not run pip install -r requirements.txt.'
    )

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('CUDA unavailable. Enable a GPU accelerator in Kaggle.')
print('GPU:', torch.cuda.get_device_name(0))
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

# Import only after the PyTorch integrity check.
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from jiwer import wer, cer
print('PyTorch and training imports: PASSED')

In [ ]:
# Locate the private Kaggle dataset. This cell is self-contained.
from pathlib import Path
import json

DATASET_CANDIDATES = [
    Path('/kaggle/input/KAGGLE/kaggle_dataset/data'),
    Path('/kaggle/input/kaggle_dataset/data'),
    Path('/kaggle/input/dataaaa/kaggle_dataset/data'),
]
DATASET_DIR = next(
    (path for path in DATASET_CANDIDATES if (path / 'splits' / 'train.jsonl').exists()),
    None,
)
if DATASET_DIR is None:
    found = list(Path('/kaggle/input').glob('**/data/splits/train.jsonl'))
    DATASET_DIR = found[0].parent.parent if found else None
if DATASET_DIR is None:
    raise FileNotFoundError('train.jsonl not found under /kaggle/input')

AUDIO_DIR = DATASET_DIR / 'processed' / 'audio'
SOURCE_TRAIN = DATASET_DIR / 'splits' / 'train.jsonl'
SOURCE_VALIDATION = DATASET_DIR / 'splits' / 'validation.jsonl'
if not AUDIO_DIR.exists():
    raise FileNotFoundError(f'Audio directory not found: {AUDIO_DIR}')

def absolute_manifest(source, name):
    records = []
    for line in source.read_text(encoding='utf-8-sig').splitlines():
        if not line.strip():
            continue
        record = json.loads(line)
        audio = Path(record['audio'])
        if audio.parts and audio.parts[0] == 'data':
            audio = Path(*audio.parts[1:])
        record['audio'] = str(DATASET_DIR / audio)
        if not Path(record['audio']).exists():
            raise FileNotFoundError(record['audio'])
        records.append(record)
    output = Path('/kaggle/working') / name
    output.write_text(
        ''.join(json.dumps(record, ensure_ascii=False) + '\n' for record in records),
        encoding='utf-8',
    )
    return output

TRAIN_MANIFEST = absolute_manifest(SOURCE_TRAIN, 'train_absolute.jsonl')
VALIDATION_MANIFEST = (
    absolute_manifest(SOURCE_VALIDATION, 'validation_absolute.jsonl')
    if SOURCE_VALIDATION.exists()
    else None
)
print('Dataset:', DATASET_DIR)
print('Audio files:', len(list(AUDIO_DIR.glob('*.wav'))))
print('Train manifest:', TRAIN_MANIFEST)
print('Validation manifest:', VALIDATION_MANIFEST)

In [ ]:
validate = REPO_DIR / 'src' / 'data' / 'validate_dataset.py'
subprocess.run([sys.executable, str(validate), str(TRAIN_MANIFEST)], check=True)
if VALIDATION_MANIFEST is not None:
    subprocess.run([sys.executable, str(validate), str(VALIDATION_MANIFEST)], check=True)
print('Manifest validation completed.')

## Pilot training

The pilot uses at most 500 train examples and 200 steps. Its metrics are engineering checks, not scientific results.

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

OUTPUT_DIR = Path('/kaggle/working/lora_pilot')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('=' * 70)
print('STARTING WHISPER LoRA PILOT')
print('=' * 70)
print(f'Repository       : {REPO_DIR}')
print(f'Train manifest   : {TRAIN_MANIFEST}')
print(f'Validation       : {VALIDATION_MANIFEST}')
print(f'Output directory : {OUTPUT_DIR}')
print('Train limit      : 500 examples')
print('Validation limit : 100 examples')
print('Maximum steps    : 200')
print('Evaluation       : every 50 steps (generation is expensive)')
print('Device           : CUDA, one GPU for this pilot')
print('-' * 70)

required_files = [REPO_DIR / 'scripts' / 'train_lora.py', Path(TRAIN_MANIFEST)]
missing = [str(path) for path in required_files if not path.exists()]
if VALIDATION_MANIFEST is not None and not Path(VALIDATION_MANIFEST).exists():
    missing.append(str(VALIDATION_MANIFEST))
if missing:
    raise FileNotFoundError('Missing required file(s): ' + ', '.join(missing))

command = [
    sys.executable, str(REPO_DIR / 'scripts' / 'train_lora.py'),
    '--train-manifest', str(TRAIN_MANIFEST),
    '--output-dir', str(OUTPUT_DIR),
    '--device', 'cuda',
    '--max-train-samples', '500',
    '--max-validation-samples', '100',
    '--max-steps', '200',
    '--logging-steps', '10',
    '--eval-steps', '50',
    '--save-steps', '50',
]
if VALIDATION_MANIFEST is not None:
    command += ['--validation-manifest', str(VALIDATION_MANIFEST)]

print('Exact command:')
print(' '.join(command))
print('-' * 70)
print('STATUS: training is starting now...')
print('Expected progress: 0/200 -> 200/200; evaluation at 50, 100, 150, 200.')
print('=' * 70)

# Restrict this pilot to one visible GPU; use Accelerate for future multi-GPU runs.
env = os.environ.copy()
env['CUDA_VISIBLE_DEVICES'] = '0'
started_at = time.time()
completed = subprocess.run(command, check=False, env=env)
elapsed_minutes = (time.time() - started_at) / 60

print('=' * 70)
if completed.returncode == 0:
    print(f'STATUS: training completed successfully in {elapsed_minutes:.1f} minutes.')
else:
    print(f'STATUS: training failed with exit code {completed.returncode} after {elapsed_minutes:.1f} minutes.')

output_files = [path for path in sorted(OUTPUT_DIR.rglob('*')) if path.is_file()]
print(f'Output directory exists: {OUTPUT_DIR.exists()}')
print(f'Output files currently present: {len(output_files)}')
for path in output_files[:20]:
    print(f'  - {path.relative_to(OUTPUT_DIR)}')
if len(output_files) > 20:
    print(f'  ... and {len(output_files) - 20} more files')

if completed.returncode != 0:
    raise subprocess.CalledProcessError(completed.returncode, command)

In [ ]:
import shutil

metrics = OUTPUT_DIR / 'metrics.json'
if metrics.exists():
    print(metrics.read_text(encoding='utf-8'))
archive = shutil.make_archive('/kaggle/working/lora_pilot_outputs', 'zip', OUTPUT_DIR)
print('Download from Kaggle Output:', archive)

## Resume training

Add `--resume-from-checkpoint /kaggle/working/lora_pilot/checkpoints/checkpoint-200` to the pilot command and rerun it. Only LoRA adapters and trainer checkpoints are saved.